In [ ]:
import uproot
import awkward as ak

import pandas as pd

import matplotlib.pylab as plt
import numpy as np

import hist
from hist import Hist

import time

import myPIDselector

import math

import os

data = ak.from_parquet("Background_and_signal_SP_modes_All_runs.parquet")

plt.close()





###################################################################################
def calculate_bits_for_PID_selector(trkidx, trk_selector_map, verbose=0):
    
    bits = None

    # If there is no trk index passed in, just calculate the bits for
    # all of the tracks
    if trkidx is None:
        trkidx = ak.local_index(trk_selector_map)

    # Grab the tracks that map on to the particle/collection we are interested in 
    subset_of_trk_selector_map = trk_selector_map[trkidx]
    if verbose:
        print("values in the subset of the trk selector map")
        print(subset_of_trk_selector_map)
        print()
        
    # We need this to convert the numbers in the selector to binary
    binary_repr_vec = np.vectorize(np.binary_repr)

    # Grab the number of entries in each so we can unflatten this later
    counts = ak.num(subset_of_trk_selector_map)
    
    # Now get the binary representation (as a string) for the flattened subset
    binrep = binary_repr_vec(ak.flatten(subset_of_trk_selector_map), width=32)

    if verbose:
        print("binary representation of selector map")
        print(binrep)
        print()

    # Convert the string to integers
    tempbits = np.array(binrep).astype(int)
    bits = ak.unflatten(tempbits,counts)

    if verbose:
        print("flattened integer representation of selectors as binary (int)")
        print(tempbits)
        print()
        print("unflattened integer representation of selectors as binary (int)")
        print(bits)
        print()

    return bits

##########################################################################

##########################################################################

def mask_PID_selection(bits, selector, pid_map_object):

    bit_to_look_for = pid_map_object.selectors.index(selector)

    place = int(math.pow(10,bit_to_look_for))
    #print(place)

    mask = bits // place % 10

    mask_bool = ak.values_astype(mask,bool)

    return mask_bool



def get_info_for_PID_masks(ak_arr, verbosity=0):

    # Proton and pion information from the Lambda decay
    # These are the index of the proton (d1) and pion (d2) in those lists
    L1d1idx = ak_arr.Lambda0d1Idx[ak_arr.Bd1Idx]
    L1d2idx = ak_arr.Lambda0d2Idx[ak_arr.Bd1Idx]

    L2d1idx = ak_arr.Lambda0d1Idx[ak_arr.Bd2Idx]
    L2d2idx = ak_arr.Lambda0d2Idx[ak_arr.Bd2Idx]


    
     
    #d1lund = ak_arr['Lambda0d1Lund']
    #d2lund = ak_arr['Lambda0d2Lund']
    
    #Bd2idx = ak_arr['Bd2Idx']
    #Bd2lund = ak_arr['Bd2Lund']
    
    trkidx_proton = data['pTrkIdx']
    trk_selector_map_proton = data['pSelectorsMap']
    
    trkidx_pion = data['piTrkIdx']
    trk_selector_map_pion = data['piSelectorsMap']

    indices_and_maps = {}
    indices_and_maps['L1d1idx'] = L1d1idx
    indices_and_maps['L1d2idx'] = L1d2idx
    indices_and_maps['L2d1idx'] = L2d1idx
    indices_and_maps['L2d2idx'] = L2d2idx
    #indices_and_maps['Bd2idx'] = Bd2idx

    indices_and_maps['trkidx_proton'] = trkidx_proton
    indices_and_maps['trk_selector_map_proton'] = trk_selector_map_proton

    indices_and_maps['trkidx_pion'] = trkidx_pion
    indices_and_maps['trk_selector_map_pion'] = trk_selector_map_pion

    return indices_and_maps


def PID_masks(indices_and_maps, \
              lam1p_selector,\
              lam1pi_selector, \
              lam2p_selector,\
              lam2pi_selector, \
             verbosity=0):
                #change Bpselector
    L1d1idx = indices_and_maps['L1d1idx']
    L1d2idx = indices_and_maps['L1d2idx']
    L2d1idx = indices_and_maps['L2d1idx']
    L2d2idx = indices_and_maps['L2d2idx']
    #Bd2idx = indices_and_maps['Bd2idx']

    trkidx_proton = indices_and_maps['trkidx_proton']
    trk_selector_map_proton = indices_and_maps['trk_selector_map_proton']

    trkidx_pion = indices_and_maps['trkidx_pion']
    trk_selector_map_pion = indices_and_maps['trk_selector_map_pion']
    
    # Proton
    pbits = calculate_bits_for_PID_selector(trkidx_proton, trk_selector_map_proton, verbose=verbosity)
    # Pion
    pibits = calculate_bits_for_PID_selector(trkidx_pion, trk_selector_map_pion, verbose=verbosity)
    
    
    #selector_proton = 'TightKMProtonSelection'
    #selector_pion = 'TightKMPionMicroSelection'
    #print(f"Now trying to create a mask with {selector_proton}")
    #print(f"Now trying to create a mask with {selector_pion}")
    
    
    mask_bool_protonL1 = mask_PID_selection(pbits[L1d1idx], lam1p_selector, pps)
        
    mask_bool_pionL1 = mask_PID_selection(pibits[L1d2idx], lam1pi_selector, pips)

    mask_bool_protonL2 = mask_PID_selection(pbits[L2d1idx], lam2p_selector, pps)
        
    mask_bool_pionL2 = mask_PID_selection(pibits[L2d2idx], lam2pi_selector, pips)

    return mask_bool_protonL1, mask_bool_pionL1,mask_bool_protonL2, mask_bool_pionL2



pps = myPIDselector.PIDselector("p")
pips = myPIDselector.PIDselector("pi")

print(data.fields)


for myPMask in ["SuperLooseKMProtonSelection"]:
    for myPiMask in ["SuperLooseKMPionMicroSelection"]: 

        indices_and_maps = get_info_for_PID_masks(data, verbosity=0)

        mask_bool_protonL1, mask_bool_pionL1,mask_bool_protonL2, mask_bool_pionL2 = PID_masks(indices_and_maps, \
                            lam1p_selector=myPMask, \
                            lam1pi_selector=myPiMask, \
                            lam2p_selector=myPMask, \
                            lam2pi_selector=myPiMask, \
                            verbosity=0)
        #for valu in ['998','1005','-999']:
        for valu in ['1005']:
            for lengthCut in [0]:
                valid = []
                inValid = []
                passCount = 0
                valCount = 0
                inValCount=0
                emptyCount=0
                spmask = (data['spmode']==valu)

                #mes = data['BpostFitMes']
                #de = data['BpostFitDeltaE']

                #meslo = INT32_MIN
                #meshi =INT32_MAX
                #DeltaElo = INT32_MIN
                #DeltaEhi = INT32_MAX
                mask_pid = mask_bool_pionL1 & mask_bool_protonL1 & mask_bool_pionL2 & mask_bool_protonL2 
                mask_pid_event = ak.any(mask_pid, axis=1)
                mask_fit = mask_pid_event #(mes>meslo) & (mes<meshi) & (de>DeltaElo) & (de<DeltaEhi)
                #mask = mask & mask_fit
                mask_fit = mask_fit[spmask]


                lamconflsig = data[spmask]['Lambda0postFitFlightSignificance'][mask_fit]
                lamfl_basic = data[spmask]['Lambda0FlightLen'][mask_fit]
                lamuncmass = data[spmask]['Lambda0_unc_Mass'][mask_fit]
                numBary = data[spmask]['nB'][mask_fit]

                #print(data.fields)

                peaks = []
                allentries = []
                sigs = []
                bkgs = []
                pcts = []
                cuts = []
                tags = []

                flight_tag = 'lampostfitsig'

                lammass_world_average = 1.115683
                width = 0.003
                lo = lammass_world_average - width
                hi = lammass_world_average + width

                nrows = 8
                ncols = 8
                
                fig1,axes1 = plt.subplots(figsize=(12,8),nrows=nrows, ncols=ncols, sharex=True, sharey=True)
                
                #for i in range(60):
                for i in [5, 10, 20, 30]:

                    #mothIdx, d1Lund

                    m = lamuncmass

                    #flight significance cut
                    mask_fl = (lamconflsig>i)&(numBary==1)

                    """

                    lamidx = data[spmask]['Lambda0MCIdx'][mask_fit]
                    lamd1idx = data[spmask]['Lambda0d1Idx'][mask_fit]
                    lamd2idx = data[spmask]['Lambda0d2Idx'][mask_fit]
                    pridx = data[spmask]['pMCIdx'][mask_fit]
                    piidx = data[spmask]['piMCIdx'][mask_fit]
                    mcLund = data[spmask]['mcLund'][mask_fit]
                    mothIdx = data[spmask]['mothIdx'][mask_fit]

                    realLambdaNotNeg = 0
                    negOneCount = 0
                    notNegPID = []
                    pidCollection = []
                    motherList = []
                    justMom = []
                    sameMotherCount =0
                    
                    #{np.int32(-413), np.int32(421), np.int32(-411), np.int32(10022), np.int32(-213), 
                    # np.int32(2224), np.int32(22), np.int32(313), np.int32(4122), np.int32(-421), np.int32(413)
                    motherDict = {}
                    sameMotherDict = {}
                    #
                    count = 0
                    neg1Count = 0
                    for indx in range(len(lamidx)):
                        lamSubL = lamidx[indx]
                        for j in range(len(lamSubL)):
                            count+=1
                            val = lamSubL[j]
                            if val!=-1:
                                pid = mcLund[indx][val]
                                if abs(pid) == 3122:
                                    realLambdaNotNeg += 1
                                else:
                                    notNegPID += [pid]
                            else:
                                neg1Count +=1
                                d1 = lamd1idx[indx][j]
                                d2 = lamd2idx[indx][j]
                                indPro = pridx[indx][d1]
                                indPi = piidx[indx][d2]

                                if indPro!=-1 and indPi != -1:
                                    pidPro = mcLund[indx][indPro]
                                    pidPi = mcLund[indx][indPi]
                                    pidCollection += [(pidPro, pidPi, indx, j)]
                                    mom1Lund =mcLund[indx][mothIdx[indx][indPro]]
                                    mom2Lund = mcLund[indx][mothIdx[indx][indPi]]
                                    motherList += [(mom1Lund, indx, j)]
                                    motherList += [(mom2Lund, indx, j)]
                                    justMom += [mom1Lund, mom2Lund]
                                    
                                    
                                    if mothIdx[indx][indPro] == mothIdx[indx][indPi]:
                                        if mom1Lund in motherDict.keys():
                                            motherDict[mom1Lund]+=1
                                        else:
                                            motherDict[mom1Lund]=1
                                        sameMotherCount +=1
                                        if mom1Lund in sameMotherDict.keys():
                                            sameMotherDict[mom1Lund]+=1
                                        else:
                                            sameMotherDict[mom1Lund]=1
                                    else:
                                        if mom1Lund in motherDict.keys():
                                            motherDict[mom1Lund]+=1
                                        else:
                                            motherDict[mom1Lund]=1
                                        if mom2Lund in motherDict.keys():
                                            motherDict[mom2Lund]+=1
                                        else:
                                            motherDict[mom2Lund]=1
                                else:
                                    negOneCount += 1
                                
                    print("Pro or Pi = -1 Count:", negOneCount)
                    print("Number of \"lambdas\": ", count)
                    print("Real Valid Lambda Count",realLambdaNotNeg)
                    print("notNegPID:", notNegPID)
                    print("Particles confused for lambdas:", set(notNegPID))
                    for val in set(notNegPID):
                        print(f"Count that {val} is confused for lambda:", notNegPID.count(val))
                    #print(pidCollection)
                    #{np.int32(-413), np.int32(421), np.int32(-411), np.int32(10022), np.int32(-213), 
                    # np.int32(2224), np.int32(22), np.int32(313), np.int32(4122), np.int32(-421), np.int32(413)}
                    print("-1 Lambda Count:", neg1Count)
                    sorted_motherdict = dict(sorted(motherDict.items(), key=lambda item: item[1], reverse=True))
                    for key in sorted_motherdict:
                        if sorted_motherdict[key]>50:
                            print(f"Mother of particle reconstructed into fake lambda = {key}")
                            print("Count of times this happened:", sorted_motherdict[key])


                    print("SAME MOTHER STUFF")
                    print("Count that two particles mis-reconstructed into lambda have same mother", sameMotherCount)
                    sorted_sameMotherdict = dict(sorted(sameMotherDict.items(), key=lambda item: item[1], reverse=True))
                    for key in sorted_sameMotherdict:
                        if sorted_sameMotherdict[key]>20:
                            print(f"Mother of particle reconstructed into fake lambda = {key}")
                            print("Count of times this happened:", sorted_sameMotherdict[key])

                    """
                
                    


                    

                    """                    

                    lamd1Lund = data[spmask]['Lambda0d1Lund'][mask_fit]
                    lamd2Lund = data[spmask]['Lambda0d2Lund'][mask_fit]
                    mothIdx = data[spmask]['mothIdx'][mask_fit]
                    print(mcLund[0][mothIdx[0][33]], mcLund[0][mothIdx[0][17]])
                    print(mcLund[0][10], mcLund[0][lamidx[0][0]])
                    print(mcLund[0][mothIdx[0][pridx[0][lamd1idx[0][0]]]])
                    
                    print(lamidx[:10])
                    print(lamd1idx[:10])
                    print(lamd2idx[:10])
                    
                    print(pridx[:10])
                    print(piidx[:10])
                    
                    print(mcLund[0][33], mcLund[0][17], mcLund[0][10])
                    """


                    #flight length cut
                    #mask_fl_length = (lamfl_basic>lengthCut)
                    """
                    mask_easy = (lamconflsig>0)
                    easyFLCount = 0
                    not10TrimCount = 0
                    flightTest=[]
                    flightnot10 = []
                    pTTest = []
                    pTnot10 = []
                    enot10 = []
                    p3not10 = []
                    flLnot10 = []
                    invMnot10 = []
                    delRnot10 = []
                    eTest = []
                    p3Test = []
                    flLTest = []
                    invMTest= []
                    delR = []
                    for ind in range(41280):
                        if all(bool for bool in mask_easy[ind]) and len(m[ind])==2:
                            if (any(not bool for bool in mask_fl[ind])) and sum(1 for indB in range(len(m[ind])) if (lo<m[ind,indB])&(m[ind,indB]<hi))>=2:
                                easyFLCount+=1
                                #print("Mask_FL, mask_easy, lamuncmass")
                                #print(mask_fl[ind])
                                #print(mask_easy[ind])
                                #print("LO",lo, "HI",hi)

                                #print("The flight sigs")
                                for val in lamconflsig[ind]:
                                    flightTest +=[val]

                                #print("Energies")
                                for val in data[spmask]['Lambda0energyCM'][mask_fit][ind]:
                                    eTest += [val]

                                #print("Flight Lengths")
                                for val in lamfl_basic[ind]:
                                    
                                    #print(val)
                                    flLTest+=[val]

                                theta1 = np.acos(data[spmask]['Lambda0costhCM'][mask_fit][ind][0])
                                theta2 = np.acos(data[spmask]['Lambda0costhCM'][mask_fit][ind][1])
                                phi1 = data[spmask]['Lambda0phiCM'][mask_fit][ind][0]
                                phi2 = data[spmask]['Lambda0phiCM'][mask_fit][ind][1]

                                if(np.tan(theta1/2) == 0):
                                    eta1 = 999999 # Effectively infinity, to avoid python evaluating log(0)
                                else:
                                    eta1 = -1*np.log(np.tan(theta1/2))

                                if(np.tan(theta2/2) == 0):
                                    eta2 = 999999 # Effectively infinity, to avoid python evaluating log(0)
                                else:
                                    eta2 = -1*np.log(np.tan(theta2/2))
                                
                                delR += [np.sqrt((phi1-phi2)**2+(eta1-eta2)**2)]

                                #print("p3CMs")
                                for val in data[spmask]['Lambda0p3CM'][mask_fit][ind]:
                                    p3Test += [val]
                                    
                                pTTest += [data[spmask]['Lambda0p3CM'][mask_fit][ind][0]*np.sin(theta1)]
                                #print("ONE / Index:", ind,":", data[spmask]['Lambda0p3CM'][mask_fit][ind][0]*np.sin(theta1))
                                pTTest += [data[spmask]['Lambda0p3CM'][mask_fit][ind][1]*np.sin(theta2)]
                                #print("Two / Index:", ind,":",data[spmask]['Lambda0p3CM'][mask_fit][ind][1]*np.sin(theta2))

                                #print("invMs")
                                for j in range(len(data[spmask]['Lambda0energyCM'][mask_fit][ind])):
                                    invMTest+=[(np.sqrt(data[spmask]['Lambda0energyCM'][mask_fit][ind][j]**2-data[spmask]['Lambda0p3CM'][mask_fit][ind][j]**2))]

                            else:
                                not10TrimCount+=1
                                #print("Mask_FL, mask_easy, lamuncmass")
                                #print(mask_fl[ind])
                                #print(mask_easy[ind])
                                #print("LO",lo, "HI",hi)

                                #print("The flight sigs")
                                for val in lamconflsig[ind]:
                                    flightnot10 +=[val]

                                #print("Energies")
                                for val in data[spmask]['Lambda0energyCM'][mask_fit][ind]:
                                    enot10 += [val]

                                #print("Flight Lengths")
                                for val in lamfl_basic[ind]:
                                    
                                    #print(val)
                                    flLnot10+=[val]

                                theta1 = np.acos(data[spmask]['Lambda0costhCM'][mask_fit][ind][0])
                                theta2 = np.acos(data[spmask]['Lambda0costhCM'][mask_fit][ind][1])
                                phi1 = data[spmask]['Lambda0phiCM'][mask_fit][ind][0]
                                phi2 = data[spmask]['Lambda0phiCM'][mask_fit][ind][1]

                                if(np.tan(theta1/2) == 0):
                                    eta1 = 999999 # Effectively infinity, to avoid python evaluating log(0)
                                else:
                                    eta1 = -1*np.log(np.tan(theta1/2))

                                if(np.tan(theta2/2) == 0):
                                    eta2 = 999999 # Effectively infinity, to avoid python evaluating log(0)
                                else:
                                    eta2 = -1*np.log(np.tan(theta2/2))
                                
                                delRnot10 += [np.sqrt((phi1-phi2)**2+(eta1-eta2)**2)]

                                #print("p3CMs")
                                for val in data[spmask]['Lambda0p3CM'][mask_fit][ind]:
                                    p3not10 += [val]
                                    
                                pTnot10 += [data[spmask]['Lambda0p3CM'][mask_fit][ind][0]*np.sin(theta1)]
                                
                                pTnot10 += [data[spmask]['Lambda0p3CM'][mask_fit][ind][1]*np.sin(theta2)]

                                #print("invMs")
                                for j in range(len(data[spmask]['Lambda0energyCM'][mask_fit][ind])):
                                    invMnot10+=[(np.sqrt(data[spmask]['Lambda0energyCM'][mask_fit][ind][j]**2-data[spmask]['Lambda0p3CM'][mask_fit][ind][j]**2))]


                    plt.figure()
                    plt.hist(flightnot10, bins=100)
                    plt.hist(flightTest, bins=100)
                    plt.xlabel(r'Flight Significance')
                    plt.figure()
                    plt.hist(flLnot10, bins=100, range=(-2, 40))
                    plt.hist(flLTest, bins=100, range=(-2, 40))
                    plt.xlabel(r'Flight Length')
                    plt.figure()
                    plt.hist(invMnot10, bins=100)
                    plt.hist(invMTest, bins=100)
                    plt.xlabel(r'Inv Mass')
                    plt.figure()
                    plt.hist(delRnot10, bins=100)
                    plt.hist(delR, bins=100)
                    plt.xlabel(r'delR')
                    plt.figure()
                    plt.hist(pTnot10, bins=100)
                    plt.hist(pTTest, bins=100)
                    plt.xlabel(r'pT')
                    
                    print("COUNT:", easyFLCount)

                    plt.show()
                    """


                    #diff cut upper and lower
                    #mask_upper_fl = (lamconflsig>i+diff)
                    #m = m[mask_fl&(ak.any(mask_upper_fl, axis=1))]

                    #only cut upper
                    #m=m[ak.any(mask_fl, axis=-1)]

                    #same cut upper and lower with flight length
                    #m = m[mask_fl&mask_fl_length]

                    #same cut upper and lower
                    m = m[mask_fl]
                    sizeMask = (ak.num(m, axis=1) == 2)

                    m = m[sizeMask]

                    outerMask = (((m[:,0]<lo) | (m[:,0]>hi)) & ((m[:,1]<lo) | (m[:,1]>hi)))
                    
                    b0 = len(m[outerMask])/4

                    s1Mask = ((m[:,0]<lo)&(lo<m[:,1])&(m[:,1]<hi))   #
                    s2Mask = ((m[:,0]>hi)&(lo<m[:,1])&(m[:,1]<hi))
                    s3Mask = ((m[:,1]<lo)&(lo<m[:,0])&(m[:,0]<hi))   #
                    s4Mask = ((m[:,1]>hi)&(lo<m[:,0])&(m[:,0]<hi))

                    #lamidx = data[spmask]['Lambda0MCIdx'][mask_fit&sizeMask]
                    if True:
                        lamidx = data[spmask]['Lambda0MCIdx'][mask_fit]
                        lamidx = lamidx[mask_fl]
                        lamidx = lamidx[sizeMask]
                        lamidxs1 = lamidx[s1Mask]
                        lamidxs3 = lamidx[s3Mask]
                        print("LEN(lamidx):",len(lamidx))
                        print("LEN(lamidxs1):",len(lamidxs1))
                        print("LEN(lamidxs3):",len(lamidxs3))
                        
                        lamd1idx = data[spmask]['Lambda0d1Idx'][mask_fit]
                        lamd1idx = lamd1idx[mask_fl]
                        lamd1idx = lamd1idx[sizeMask]
                        lamd1idxs1 = lamd1idx[s1Mask]
                        lamd1idxs3 = lamd1idx[s3Mask]
                        print("LEN(lamd1idx):",len(lamd1idx))
                        print("LEN(lamd1idxs1):",len(lamd1idxs1))
                        print("LEN(lamd1idxs3):",len(lamd1idxs3))
                        lamd2idx = data[spmask]['Lambda0d2Idx'][mask_fit]
                        lamd2idx = lamd2idx[mask_fl]
                        lamd2idx = lamd2idx[sizeMask]
                        lamd2idxs1 = lamd2idx[s1Mask]
                        lamd2idxs3 = lamd2idx[s3Mask]
                        print("LEN(lamd2idx):",len(lamd2idx))
                        print("LEN(lamd2idxs1):",len(lamd2idxs1))
                        print("LEN(lamd2idxs3):",len(lamd2idxs3))
                        pridx = data[spmask]['pMCIdx'][mask_fit]
                        pridx = pridx[mask_fl]
                        pridx = pridx[sizeMask]
                        pridxs1 = pridx[s1Mask]
                        pridxs3 = pridx[s3Mask]
                        print("LEN(PRIDX):",len(pridx))
                        print("LEN(pridxs1):",len(pridxs1))
                        print("LEN(pridxs3):",len(pridxs3))
                        piidx = data[spmask]['piMCIdx'][mask_fit]
                        piidx = piidx[mask_fl]
                        piidx = piidx[sizeMask]
                        piidxs1 = piidx[s1Mask]
                        piidxs3 = piidx[s3Mask]
                        print("LEN(PIIDX):",len(piidx))
                        print("LEN(piidxs1):",len(piidxs1))
                        print("LEN(piidxs3):",len(piidxs3))
                        mcLund = data[spmask]['mcLund'][mask_fit]
                        mcLund = mcLund[ak.all(mask_fl,axis=1)]
                        mcLunds1 = mcLund[s1Mask]
                        mcLunds3 = mcLund[s3Mask]
                        print("LEN(mcLund):", len(mcLund))
                        print("LEN(mcLunds1):",len(mcLunds1))
                        print("LEN(mcLunds3):",len(mcLunds3))
                        #mcLund = mcLund[sizeMask]
                        mothIdx = data[spmask]['mothIdx'][mask_fit]
                        mothIdx = mothIdx[ak.all(mask_fl,axis=1)]
                        print("LEN(mothIdx):", len(mothIdx))
                        mothIdxs1 = mothIdx[s1Mask]
                        mothIdxs3 = mothIdx[s3Mask]
                        print("LEN(mothIdxs1):",len(mothIdxs1))
                        print("LEN(mothIdxs3):",len(mothIdxs3))
                        #mothIdx = mothIdx[sizeMask]
                    leftSect = lamidx[s1Mask]
                    print(leftSect[:15])
                    bottSect = lamidx[s3Mask]
                    print(bottSect[:15])

                    realLambdaNotNegs1 = 0
                    negOneCounts1 = 0
                    notNegPIDs1 = []
                    pidCollections1 = []
                    motherLists1 = []
                    justMoms1 = []
                    sameMotherCounts1 =0
                    proDicts1={}
                    piDicts1 = {}
                    
                    #{np.int32(-413), np.int32(421), np.int32(-411), np.int32(10022), np.int32(-213), 
                    # np.int32(2224), np.int32(22), np.int32(313), np.int32(4122), np.int32(-421), np.int32(413)
                    motherDicts1 = {}
                    sameMotherDicts1 = {}
                    #"""
                    counts1 = 0
                    neg1Counts1 = 0
                    for indx in range(len(lamidxs1)):
                        lamSubL = lamidxs1[indx]
                        #for j in range(len(lamSubL)):
                        for j in [0]:
                            counts1+=1
                            val = lamSubL[j]
                            if val!=-1:
                                pid = mcLunds1[indx][val]
                                if abs(pid) == 3122:
                                    realLambdaNotNegs1 += 1
                                else:
                                    notNegPIDs1 += [pid]
                            if True:
                                neg1Counts1 +=1
                                d1 = lamd1idxs1[indx][j]
                                d2 = lamd2idxs1[indx][j]
                                indPro = pridxs1[indx][d1]
                                indPi = piidxs1[indx][d2]

                                if indPro!=-1 and indPi != -1:
                                    pidPro = mcLunds1[indx][indPro]
                                    pidPi = mcLunds1[indx][indPi]
                                    if pidPro in proDicts1.keys():
                                        proDicts1[pidPro]+=1
                                    else:
                                        proDicts1[pidPro]=1
                                    if pidPi in piDicts1.keys():
                                        piDicts1[pidPi]+=1
                                    else:
                                        piDicts1[pidPi]=1
                                    mom1Lund = mcLunds1[indx][mothIdxs1[indx][indPro]]
                                    mom2Lund = mcLunds1[indx][mothIdxs1[indx][indPi]]
                                    motherLists1 += [(mom1Lund, indx, j)]
                                    motherLists1 += [(mom2Lund, indx, j)]
                                    justMoms1 += [mom1Lund, mom2Lund]
                                    
                                    
                                    if mothIdxs1[indx][indPro] == mothIdxs1[indx][indPi]:
                                        if mom1Lund in motherDicts1.keys():
                                            motherDicts1[mom1Lund]+=1
                                        else:
                                            motherDicts1[mom1Lund]=1
                                        sameMotherCounts1 +=1
                                        if mom1Lund in sameMotherDicts1.keys():
                                            sameMotherDicts1[mom1Lund]+=1
                                        else:
                                            sameMotherDicts1[mom1Lund]=1
                                    else:
                                        if mom1Lund in motherDicts1.keys():
                                            motherDicts1[mom1Lund]+=1
                                        else:
                                            motherDicts1[mom1Lund]=1
                                        if mom2Lund in motherDicts1.keys():
                                            motherDicts1[mom2Lund]+=1
                                        else:
                                            motherDicts1[mom2Lund]=1
                                else:
                                    negOneCounts1 += 1
                    
                    print("S1 S1 S1 S1 S1 S1")
                    print("Pro or Pi = -1 Count:", negOneCounts1)
                    print("Number of \"lambdas\": ", counts1)
                    print("Real Valid Lambda Count",realLambdaNotNegs1)
                    print("notNegPID:", notNegPIDs1)
                    print("Particles confused for lambdas:", set(notNegPIDs1))
                    for val in set(notNegPIDs1):
                        print(f"Count that {val} is confused for lambda:", notNegPIDs1.count(val))
                    #print(pidCollection)
                    #{np.int32(-413), np.int32(421), np.int32(-411), np.int32(10022), np.int32(-213), 
                    # np.int32(2224), np.int32(22), np.int32(313), np.int32(4122), np.int32(-421), np.int32(413)}
                    print("-1 Lambda Count:", neg1Counts1)
                    sorted_motherdicts1 = dict(sorted(motherDicts1.items(), key=lambda item: item[1], reverse=True))
                    for key in sorted_motherdicts1:
                        if sorted_motherdicts1[key]>30:
                            print(f"Mother of particle reconstructed into fake lambda = {key}")
                            print("Count of times this happened:", sorted_motherdicts1[key])


                    print("SAME MOTHER STUFF")
                    print("Count that two particles mis-reconstructed into lambda have same mother", sameMotherCounts1)
                    sorted_sameMotherdicts1 = dict(sorted(sameMotherDicts1.items(), key=lambda item: item[1], reverse=True))
                    for key in sorted_sameMotherdicts1:
                        if sorted_sameMotherdicts1[key]>10:
                            print(f"Mother of particle reconstructed into fake lambda = {key}")
                            print("Count of times this happened:", sorted_sameMotherdicts1[key])

                    print("PROTONS")
                    sorted_prodicts1 = dict(sorted(proDicts1.items(), key=lambda item: item[1], reverse=True))
                    for key in sorted_prodicts1:
                        if True:
                            print(f"PID of proton daughter of lambda = {key}")
                            print("Count of times this happened:", sorted_prodicts1[key])

                    print("PIONS")
                    sorted_pidicts1 = dict(sorted(piDicts1.items(), key=lambda item: item[1], reverse=True))
                    for key in sorted_pidicts1:
                        if True:
                            print(f"PID of proton daughter of lambda = {key}")
                            print("Count of times this happened:", sorted_pidicts1[key])




                    realLambdaNotNegs3 = 0
                    negOneCounts3 = 0
                    notNegPIDs3 = []
                    pidCollections3 = []
                    motherLists3 = []
                    justMoms3 = []
                    sameMotherCounts3 =0
                    
                    #{np.int32(-413), np.int32(421), np.int32(-411), np.int32(10022), np.int32(-213), 
                    # np.int32(2224), np.int32(22), np.int32(313), np.int32(4122), np.int32(-421), np.int32(413)
                    proDicts3={}
                    piDicts3={}
                    motherDicts3 = {}
                    sameMotherDicts3 = {}
                    #"""
                    counts3 = 0
                    neg1Counts3 = 0
                    for indx in range(len(lamidxs3)):
                        lamSubL = lamidxs3[indx]
                        #for j in range(len(lamSubL)):
                        for j in [1]:
                            counts3+=1
                            val = lamSubL[j]
                            if val!=-1:
                                pid = mcLunds3[indx][val]
                                if abs(pid) == 3122:
                                    realLambdaNotNegs3 += 1
                                else:
                                    notNegPIDs3 += [pid]
                            if True:
                                neg1Counts3 +=1
                                d1 = lamd1idxs3[indx][j]
                                d2 = lamd2idxs3[indx][j]
                                indPro = pridxs3[indx][d1]
                                indPi = piidxs3[indx][d2]

                                if indPro!=-1 and indPi != -1:
                                    pidPro = mcLunds3[indx][indPro]
                                    pidPi = mcLunds3[indx][indPi]
                                    if pidPro in proDicts3.keys():
                                        proDicts3[pidPro]+=1
                                    else:
                                        proDicts3[pidPro]=1
                                    if pidPi in piDicts3.keys():
                                        piDicts3[pidPi]+=1
                                    else:
                                        piDicts3[pidPi]=1
                                    
                                    mom1Lund =mcLunds3[indx][mothIdxs3[indx][indPro]]
                                    mom2Lund = mcLunds3[indx][mothIdxs3[indx][indPi]]
                                    motherLists3 += [(mom1Lund, indx, j)]
                                    motherLists3 += [(mom2Lund, indx, j)]
                                    justMoms3 += [mom1Lund, mom2Lund]
                                    
                                    
                                    if mothIdxs3[indx][indPro] == mothIdxs3[indx][indPi]:
                                        if mom1Lund in motherDicts3.keys():
                                            motherDicts3[mom1Lund]+=1
                                        else:
                                            motherDicts3[mom1Lund]=1
                                        sameMotherCounts3 +=1
                                        if mom1Lund in sameMotherDicts3.keys():
                                            sameMotherDicts3[mom1Lund]+=1
                                        else:
                                            sameMotherDicts3[mom1Lund]=1
                                    else:
                                        if mom1Lund in motherDicts3.keys():
                                            motherDicts3[mom1Lund]+=1
                                        else:
                                            motherDicts3[mom1Lund]=1
                                        if mom2Lund in motherDicts3.keys():
                                            motherDicts3[mom2Lund]+=1
                                        else:
                                            motherDicts3[mom2Lund]=1
                                else:
                                    negOneCounts3 += 1
                    
                    print("S3 S3 S3 S3 S3 S3")
                    print("Pro or Pi = -1 Count:", negOneCounts3)
                    print("Number of \"lambdas\": ", counts3)
                    print("Real Valid Lambda Count",realLambdaNotNegs3)
                    print("notNegPID:", notNegPIDs3)
                    print("Particles confused for lambdas:", set(notNegPIDs3))
                    for val in set(notNegPIDs3):
                        print(f"Count that {val} is confused for lambda:", notNegPIDs3.count(val))
                    #print(pidCollection)
                    #{np.int32(-413), np.int32(421), np.int32(-411), np.int32(10022), np.int32(-213), 
                    # np.int32(2224), np.int32(22), np.int32(313), np.int32(4122), np.int32(-421), np.int32(413)}
                    print("-1 Lambda Count:", neg1Counts3)
                    sorted_motherdicts3 = dict(sorted(motherDicts3.items(), key=lambda item: item[1], reverse=True))
                    for key in sorted_motherdicts3:
                        if sorted_motherdicts3[key]>30:
                            print(f"Mother of particle reconstructed into fake lambda = {key}")
                            print("Count of times this happened:", sorted_motherdicts3[key])


                    print("SAME MOTHER STUFF")
                    print("Count that two particles mis-reconstructed into lambda have same mother", sameMotherCounts3)
                    sorted_sameMotherdicts3 = dict(sorted(sameMotherDicts3.items(), key=lambda item: item[1], reverse=True))
                    for key in sorted_sameMotherdicts3:
                        if sorted_sameMotherdicts3[key]>10:
                            print(f"Mother of particle reconstructed into fake lambda = {key}")
                            print("Count of times this happened:", sorted_sameMotherdicts3[key])

                    print("PROTONS")
                    sorted_prodicts3 = dict(sorted(proDicts3.items(), key=lambda item: item[1], reverse=True))
                    for key in sorted_prodicts3:
                        if True:
                            print(f"PID of proton daughter of lambda = {key}")
                            print("Count of times this happened:", sorted_prodicts3[key])

                    print("PIONS")
                    sorted_pidicts3 = dict(sorted(piDicts3.items(), key=lambda item: item[1], reverse=True))
                    for key in sorted_pidicts3:
                        if True:
                            print(f"PID of proton daughter of lambda = {key}")
                            print("Count of times this happened:", sorted_pidicts3[key])



                    b1 = (len(m[s1Mask])+len(m[s2Mask])+len(m[s3Mask])+len(m[s4Mask]))/4 - b0

                    s5Mask = (((lo<m[:,0])&(m[:,0]<hi))&((lo<m[:,1])&(m[:,1]<hi)))

                

                    mass_cand_out_0 = m[outerMask][:, 0]
                    mass_cand_out_1 = m[outerMask][:, 1]

                    mass_cand_s1_0 = m[s1Mask][:, 0]
                    mass_cand_s1_1 = m[s1Mask][:, 1]

                    mass_cand_s2_0 = m[s2Mask][:, 0]
                    mass_cand_s2_1 = m[s2Mask][:, 1]

                    mass_cand_s3_0 = m[s3Mask][:, 0]
                    mass_cand_s3_1 = m[s3Mask][:, 1]

                    mass_cand_s4_0 = m[s4Mask][:, 0]
                    mass_cand_s4_1 = m[s4Mask][:, 1]

                    mass_cand_s5_0 = m[s5Mask][:, 0]
                    mass_cand_s5_1 = m[s5Mask][:, 1]

                    all_x = [mass_cand_out_0, mass_cand_s1_0, mass_cand_s2_0,mass_cand_s3_0, mass_cand_s4_0, mass_cand_s5_0]
                    all_y = [mass_cand_out_1, mass_cand_s1_1, mass_cand_s2_1,mass_cand_s3_1, mass_cand_s4_1, mass_cand_s5_1]

                    # 2. Concatenate them into single flat NumPy arrays
                    x_combined = ak.to_numpy(ak.concatenate(all_x))
                    y_combined = ak.to_numpy(ak.concatenate(all_y))

                    # 2. Create a 2D Histogram (highly recommended for large particle physics datasets)
                    if False:
                        plt.figure(figsize=(8, 6))
                        plt.hist2d(
                            x_combined, 
                            y_combined, 
                            bins=100, 
                            range=[[lo-1.05*width, hi+1.05*width], [lo-1.05*width, hi+1.05*width]], 
                            cmap='viridis'
                        )

                        # 3. Add labels and a colorbar
                        plt.colorbar(label='Candidates per bin')
                        plt.xlabel(r'$\Lambda$ Candidate 1 Mass [GeV/$c^2$]')
                        plt.ylabel(r'$\Lambda$ Candidate 2 Mass [GeV/$c^2$]')
                        plt.title('2D Mass Correlation')

                        plt.show()
                    

                    nsig = len(m[s5Mask]) - b1 - b0

                    nbkg = b1+b0
                    npeak = len(m[s5Mask])
                    nall = len(lamuncmass[lamfl_basic>=0]) #?????? Should this be events with 2 lambda, events with enough fltsig, or all events?

                    peaks.append(npeak)
                    sigs.append(nsig)
                    bkgs.append(nbkg)
                    cuts.append(i)
                    allentries.append(nall)
                    tags.append(flight_tag)

                    row = int(i/ncols)
                    col = i%ncols
                    
                    plt.sca(axes1[row][col])
                        
                    plt.hist(ak.flatten(lamuncmass, axis=None),bins=100,range=(1.105, 1.125), label=f'flcut>{i:.1f}')
                    plt.hist(ak.flatten(m, axis=None),      bins=100,range=(1.105, 1.125))
                    plt.legend()

                    #plt.sca(axes2[row][col])
                    if row == nrows-1:
                        axes1[row][col].set_xlabel(r'$\Lambda^0$ mass [GeV/c$^2$]')

                    fllo, flhi = 0,30
                    if flight_tag=='lampostfitsig':
                        fllo, flhi = 0,100


                

                sigs = np.array(sigs)
                bkgs = np.array(bkgs)
                peaks = np.array(peaks)
                cuts = np.array(cuts)
                allentries = np.array(allentries)

                pcts = sigs/sigs[0]
                bkg_under_peak = peaks - sigs
                pct_bkg_under_peak = bkg_under_peak / bkg_under_peak[0]

                S = sigs
                B = bkg_under_peak
                scale = 1
                fom = scale*S/np.sqrt(scale*S + scale*B)
                
                max_fom = max(fom)
                idx_max = fom.tolist().index(max_fom)
                print(valu, "/", myPMask, myPiMask)
                print(f"idx_max: {idx_max}  max cuts: {cuts[idx_max]:.2f} max fom: {fom[idx_max]:.2f}   pcts: {pcts[idx_max]:.2f}")
                print(f'{lengthCut} is the flight length cut')


                plt.figure(figsize=(12, 8))
                plt.subplot(3,2,1)
                plt.plot(cuts, peaks,'o')
                plt.ylabel('# under peak')
                plt.xlabel(f'Cut on flight sig value')

                plt.subplot(3,2,2)
                plt.plot(cuts, sigs,'o')
                #plt.ylabel('# of signal in peak (# in peak - # est background')
                plt.ylabel('# of signal in peak')
                plt.xlabel(f'Cut on flight sig value')

                plt.subplot(3,2,3)
                plt.plot(cuts, bkgs,'o')
                plt.ylabel('# est background')
                plt.xlabel(f'Cut on flight sig value')
                
                plt.subplot(3,2,4)
                plt.plot(cuts, pcts,'o')
                plt.ylim(0.1,1.1)
                plt.ylabel('% sig remaining')
                plt.xlabel(f'Cut on flight sig value')
                
                plt.subplot(3,2,5)
                plt.plot(cuts, pct_bkg_under_peak,'o')
                #plt.ylim(0.7,1.1)
                plt.ylabel('% bkg under peak')
                plt.xlabel(f'Cut on flight sig value')
                
                # Naive significance
                # Multiply by 10 if we are only doing Run 1
                plt.subplot(3,2,6)
                #plt.plot(cuts, 10*sigs/np.sqrt(10*bkg_under_peak),'o')
                plt.plot(cuts, fom,'o')
                plt.ylabel(r'$S/\sqrt{ S + B }$')
                plt.xlabel(f'Cut on flight sig value')

                plt.suptitle(f'MYLABEL')

                plt.tight_layout()
                
                plt.legend() 
                plt.show()
